# Specimen 02 — Prompt Engineering Experiments

Goal: same task, different prompt structures. Learn that how you ask matters as much as which model you call.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])


## 1. Pick one fixed task

e.g. extracting structured data from a messy paragraph, or classifying short text into categories.

In [2]:
MODEL = 'claude-opus-5'

TASK_DESCRIPTION = "Classify each customer support message into exactly one category: billing, technical, account, or other."

test_inputs = [
    "I was charged twice for my subscription this month, can you refund the extra charge?",
    "The app crashes every time I try to upload a photo.",
    "I can't remember my password and the reset link isn't arriving.",
    "Do you offer a student discount?",
    "My invoice shows a charge from a plan I cancelled three months ago.",
    "The website is really slow when I try to load my dashboard.",
    "I want to change the email address associated with my account.",
    "What are your business hours?",
]

true_labels = [
    "billing", "technical", "account", "other",
    "billing", "technical", "account", "other",
]

print(f'{len(test_inputs)} test inputs, task: {TASK_DESCRIPTION}')

8 test inputs, task: Classify each customer support message into exactly one category: billing, technical, account, or other.


## 2. Write 3+ prompt versions

Zero-shot vs. few-shot with examples vs. chain-of-thought vs. a version with a strict output format specified — same task, meaningfully different approaches.

In [3]:
def zero_shot_prompt(message):
    return f"Classify this customer support message into exactly one category: billing, technical, account, or other. Respond with only the category word.\n\nMessage: {message}"

def few_shot_prompt(message):
    examples = (
        "Message: I was double-billed last week.\nCategory: billing\n\n"
        "Message: The app won't open on my phone.\nCategory: technical\n\n"
        "Message: I need to update my shipping address.\nCategory: account\n\n"
    )
    return f"Classify the message into exactly one category: billing, technical, account, or other. Respond with only the category word.\n\n{examples}Message: {message}\nCategory:"

def cot_prompt(message):
    return (
        "Classify this customer support message into exactly one category: billing, technical, account, or other. "
        "First reason step by step about what the message is really asking for, then on the last line write 'Category: <word>'.\n\n"
        f"Message: {message}"
    )

def strict_format_prompt(message):
    return (
        "Classify this customer support message. Respond with ONLY a single lowercase word: "
        "billing, technical, account, or other. No punctuation, no explanation, no extra text.\n\n"
        f"Message: {message}"
    )

prompt_versions = {
    'zero_shot': zero_shot_prompt,
    'few_shot': few_shot_prompt,
    'chain_of_thought': cot_prompt,
    'strict_format': strict_format_prompt,
}
print(list(prompt_versions.keys()))

['zero_shot', 'few_shot', 'chain_of_thought', 'strict_format']


## 3. Run all versions on the same inputs

5-10 test examples, same model and temperature, so the prompt is the only variable that changes.

In [4]:
import re

def extract_category(text):
    match = re.search(r'category:\s*(billing|technical|account|other)\b', text, re.IGNORECASE)
    if match:
        return match.group(1).lower()
    match = re.search(r'\b(billing|technical|account|other)\b', text.lower())
    return match.group(1) if match else None

results = {name: [] for name in prompt_versions}

for name, builder in prompt_versions.items():
    for message in test_inputs:
        response = client.messages.create(
            model=MODEL,
            thinking={"type": "disabled"},
            max_tokens=700,
            messages=[{"role": "user", "content": builder(message)}],
        )
        text = ''.join(b.text for b in response.content if b.type == 'text')
        results[name].append(extract_category(text))
    print(f'{name}: done')

zero_shot: done


few_shot: done


chain_of_thought: done


strict_format: done


## 4. Score objectively

Define a pass/fail or scoring rule per input *before* looking at outputs. Grading after the fact is how you talk yourself into your favorite prompt winning.

In [5]:
def score(predictions, labels):
    correct = sum(p == l for p, l in zip(predictions, labels))
    return correct / len(labels)

print('Scoring rule: exact category match against the fixed labels defined in step 1, decided before any output was seen.\n')

for name, predictions in results.items():
    accuracy = score(predictions, true_labels)
    print(f'{name}: {accuracy:.1%}  {predictions}')

Scoring rule: exact category match against the fixed labels defined in step 1, decided before any output was seen.

zero_shot: 100.0%  ['billing', 'technical', 'account', 'other', 'billing', 'technical', 'account', 'other']
few_shot: 87.5%  ['billing', 'technical', 'account', 'billing', 'billing', 'technical', 'account', 'other']
chain_of_thought: 87.5%  ['billing', 'technical', 'account', 'billing', 'billing', 'technical', 'account', 'other']
strict_format: 87.5%  ['billing', 'technical', 'account', 'billing', 'billing', 'technical', 'account', 'other']


## 5. Try structured output

Use JSON mode / tool-schema to force one prompt version into strictly-typed output. Check how often it actually validates against your schema.

In [6]:
import json

schema = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "category": {"type": "string", "enum": ["billing", "technical", "account", "other"]}
        },
        "required": ["category"],
        "additionalProperties": False
    }
}

schema_response = client.messages.create(
    model=MODEL,
    thinking={"type": "disabled"},
    max_tokens=200,
    messages=[{"role": "user", "content": f"Classify this customer support message.\n\nMessage: {test_inputs[0]}"}],
    output_config={"format": schema},
)

structured_text = ''.join(b.text for b in schema_response.content if b.type == 'text')
print('Raw structured output:', structured_text)
parsed = json.loads(structured_text)
print('Parsed category:', parsed['category'])

schema_valid_count = 0
for message in test_inputs:
    r = client.messages.create(
        model=MODEL,
        thinking={"type": "disabled"},
        max_tokens=200,
        messages=[{"role": "user", "content": f"Classify this customer support message.\n\nMessage: {message}"}],
        output_config={"format": schema},
    )
    t = ''.join(b.text for b in r.content if b.type == 'text')
    try:
        json.loads(t)
        schema_valid_count += 1
    except json.JSONDecodeError:
        pass

print(f'\nSchema validated on {schema_valid_count}/{len(test_inputs)} calls')

Raw structured output: {"category":"billing"}
Parsed category: billing



Schema validated on 8/8 calls


## 6. Write up findings

Which version won, and your best guess why — specificity? examples? output constraints?

In [7]:
best_name = max(results, key=lambda n: score(results[n], true_labels))
best_score = score(results[best_name], true_labels)

summary = f'''Best-performing prompt: {best_name} at {best_score:.1%} accuracy on {len(test_inputs)} fixed test messages.
Structured output (JSON schema) validated on {schema_valid_count}/{len(test_inputs)} calls, trading a small amount of flexibility for guaranteed parseable output.
Scoring rule (exact category match) was fixed before any prompt was run, so this comparison reflects the prompts, not post-hoc judgment.'''

print(summary)

Best-performing prompt: zero_shot at 100.0% accuracy on 8 fixed test messages.
Structured output (JSON schema) validated on 8/8 calls, trading a small amount of flexibility for guaranteed parseable output.
Scoring rule (exact category match) was fixed before any prompt was run, so this comparison reflects the prompts, not post-hoc judgment.
